# Lab 10 — Final Data Validation

**WHAT:** Validate Federation and incremental CDC results.

**WHY:** Confirm source/target consistency, correct change history,
valid checkpoints, and absence of duplicate customer IDs.

**INPUT:** Delta CDC source, target, history, and checkpoint.

**OUTPUT:** PASS/FAIL validation report.

**KEY CONCEPTS:** Data reconciliation, idempotency, auditability,
and version-based incremental processing.

**RERUN SAFETY:** Read-only. Does not modify any tables.

In [0]:

BASE = "workspace.lab_10_federation_cdc"

SOURCE = f"{BASE}.customers_delta"
TARGET = f"{BASE}.customers_target"
HISTORY = f"{BASE}.customer_change_history"
CHECKPOINT = f"{BASE}.cdc_checkpoint"

PIPELINE = "customers_delta_to_target"

# 1. Validate checkpoint
checkpoint_rows = spark.sql(f"""
    SELECT last_processed_version
    FROM {CHECKPOINT}
    WHERE pipeline_name = '{PIPELINE}'
""").collect()

assert len(checkpoint_rows) == 1, "Invalid checkpoint rows"

checkpoint = int(checkpoint_rows[0]["last_processed_version"])

latest = int(
    spark.sql(f"DESCRIBE HISTORY {SOURCE} LIMIT 1")
    .first()["version"]
)

assert checkpoint <= latest, "Checkpoint exceeds source version"

# 2. Validate customer uniqueness
for table in [SOURCE, TARGET]:
    dup_count = spark.sql(f"""
        SELECT COUNT(*) AS n
        FROM (
            SELECT customer_id
            FROM {table}
            GROUP BY customer_id
            HAVING COUNT(*) > 1
        )
    """).first()["n"]

    assert dup_count == 0, f"Duplicate customers: {table}"

# 3. Compare target with source at the processed version
left_diff = spark.sql(f"""
    SELECT COUNT(*) AS n
    FROM (
        SELECT * FROM {SOURCE} VERSION AS OF {checkpoint}
        EXCEPT ALL
        SELECT * FROM {TARGET}
    )
""").first()["n"]

right_diff = spark.sql(f"""
    SELECT COUNT(*) AS n
    FROM (
        SELECT * FROM {TARGET}
        EXCEPT ALL
        SELECT * FROM {SOURCE} VERSION AS OF {checkpoint}
    )
""").first()["n"]

assert left_diff == 0 and right_diff == 0, (
    "Source and target do not match at checkpoint"
)

# 4. Validate history uniqueness
history_duplicates = spark.sql(f"""
    SELECT COUNT(*) AS n
    FROM (
        SELECT customer_id, _commit_version, _change_type
        FROM {HISTORY}
        GROUP BY customer_id, _commit_version, _change_type
        HAVING COUNT(*) > 1
    )
""").first()["n"]

assert history_duplicates == 0, "Duplicate audit events"

# 5. Compare persisted audit to the original CDF events
audit_start = spark.sql(f"""
    SELECT MIN(_commit_version) AS v
    FROM {HISTORY}
""").first()["v"]

if audit_start is not None:
    columns = """
        customer_id, first_name, last_name,
        email, city, country, signup_date,
        _change_type, _commit_version, _commit_timestamp
    """

    original_events = spark.sql(f"""
        SELECT {columns}
        FROM table_changes(
            '{SOURCE}', {int(audit_start)}, {checkpoint}
        )
    """)

    recorded_events = spark.sql(f"""
        SELECT {columns}
        FROM {HISTORY}
        WHERE _commit_version <= {checkpoint}
    """)

    assert original_events.exceptAll(
        recorded_events
    ).count() == 0, "CDF events missing from history"

    assert recorded_events.exceptAll(
        original_events
    ).count() == 0, "Unexpected events in history"

# 6. Report
source_count = spark.sql(
    f"SELECT COUNT(*) AS n FROM {SOURCE}"
).first()["n"]

target_count = spark.sql(
    f"SELECT COUNT(*) AS n FROM {TARGET}"
).first()["n"]

history_count = spark.sql(
    f"SELECT COUNT(*) AS n FROM {HISTORY}"
).first()["n"]

print("LAB 10 - FINAL VALIDATION")
print("--------------------------")
print(f"Source rows:           {source_count}")
print(f"Target rows:           {target_count}")
print(f"Historical events:     {history_count}")
print(f"Processed version:     {checkpoint}")
print(f"Latest source version: {latest}")
print(f"Source/target diffs:   {left_diff + right_diff}")
print(f"History duplicates:    {history_duplicates}")
print("--------------------------")
print("PASS: All validations successful")
